# Q7 - Train and save the churn model

Goal: a small model that predicts `still_active` from the **aggregated player table only** (`player_features.parquet`, one row per player): no match-level / timestamp table is needed to use it.

Choices (from notebook 06):
- Random forest: best or tied in every feature set, stable, no scaling needed.
- **12 inputs** instead of 68: recency, volume and a few behaviour features that a person can type or read from a player profile. On 840 players, 12 features score the same as 68 (0.83 vs 0.82 CV ROC-AUC, all inside the fold noise); the recent-window features of notebook 06 added nothing and are not used.
- The model returns the probability that the player is **still active**; `churn risk = 1 - probability`. The decision threshold on the churn risk is **0.5** (see section 2 for the trade-off).
- The final model is fitted on all 840 players; the quality numbers below come from cross-validation and a holdout, **not** from the final fit.
- Caveat: `still_active` is a noisy label (91 "active" players are silent for 90+ days), so a perfect score is not possible.

In [1]:
from datetime import date
from pathlib import Path

import joblib
import numpy as np
import pandas as pd
import polars as pl
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, precision_recall_curve, roc_auc_score
from sklearn.model_selection import RepeatedStratifiedKFold, StratifiedKFold, cross_val_predict, cross_validate, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import QuantileTransformer

FEATURES_PATH: Path = Path('player_features.parquet')
MODEL_PATH: Path = Path('churn_model.joblib')
TARGET: str = 'still_active'
SEED: int = 42
TEST_SIZE: float = 0.2

# name -> (label, unit / meaning), in the order they appear in the app
FEATURE_INFO: dict[str, tuple[str, str]] = {
    'days_since_last_match': ('Days since last match', 'days, as of today'),
    'matches_last_30d': ('Matches in the last 30 days', 'count'),
    'matches_last_90d': ('Matches in the last 90 days', 'count'),
    'matches': ('Matches played (lifetime)', 'count'),
    'tenure_days': ('Tenure', 'days between first and last match'),
    'sessions_per_week': ('Sessions per week', 'a session = matches less than 30 min apart'),
    'share_night': ('Share of matches at night (UTC 0-6h)', '0 to 1'),
    'stay_after_loss': ('Requeue rate after a loss', '0 to 1, share of losses followed by another match in the same session'),
    'share_pvp_ranked': ('Share of ranked matches', '0 to 1'),
    'n_modes': ('Number of game modes played', 'count'),
    'win_rate': ('Win rate', '0 to 1'),
    'kd_ratio': ('K/D ratio', 'kills / deaths'),
}
FEATURES: list[str] = list(FEATURE_INFO)

df_players: pl.DataFrame = pl.read_parquet(FEATURES_PATH)
X: pd.DataFrame = df_players.select(FEATURES).to_pandas()
y: np.ndarray = df_players[TARGET].to_numpy().astype(int)
print(X.shape, '| active share:', round(y.mean(), 3))
X.describe().T.round(2)

FileNotFoundError: Le fichier spécifié est introuvable. (os error 2): player_features.parquet

This error occurred with the following context stack:
	[1] 'parquet scan'
	[2] 'sink'


## 1. Evaluate before saving

Repeated stratified 5-fold CV (3 repeats) on the training part and a stratified 20% holdout, for a dummy baseline, the single feature `days_since_last_match`, logistic regression and the random forest.

In [ ]:
def make_rf() -> RandomForestClassifier:
    return RandomForestClassifier(n_estimators=500, min_samples_leaf=5, n_jobs=-1, random_state=SEED)


MODELS: dict = {
    'dummy (prior)': (make_pipeline(DummyClassifier(strategy='prior')), FEATURES),
    '1 feature (days_since_last_match)': (make_pipeline(SimpleImputer(strategy='median'), QuantileTransformer(n_quantiles=100, output_distribution='normal'), LogisticRegression()), ['days_since_last_match']),
    'logistic regression': (make_pipeline(SimpleImputer(strategy='median'), QuantileTransformer(n_quantiles=100, output_distribution='normal'), LogisticRegression(C=0.1, max_iter=2000)), FEATURES),
    'random forest': (make_pipeline(SimpleImputer(strategy='median'), make_rf()), FEATURES),
}
idx_tr, idx_te = train_test_split(np.arange(len(y)), test_size=TEST_SIZE, stratify=y, random_state=SEED)
cv = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=SEED)
rows = []
for name, (model, cols) in MODELS.items():
    res = cross_validate(model, X.iloc[idx_tr][cols], y[idx_tr], cv=cv, scoring=['roc_auc', 'average_precision'])
    model.fit(X.iloc[idx_tr][cols], y[idx_tr])
    p = model.predict_proba(X.iloc[idx_te][cols])[:, 1]
    rows.append({
        'model': name,
        'cv_roc_auc': f"{res['test_roc_auc'].mean():.3f} +/- {res['test_roc_auc'].std():.3f}",
        'cv_pr_auc': f"{res['test_average_precision'].mean():.3f} +/- {res['test_average_precision'].std():.3f}",
        'holdout_roc_auc': round(roc_auc_score(y[idx_te], p), 3),
        'holdout_pr_auc': round(average_precision_score(y[idx_te], p), 3),
    })
df_eval = pd.DataFrame(rows)
df_eval

## 2. Decision threshold (churn class)

On out-of-fold predictions of the random forest (all players), the precision / recall trade-off at several churn-risk thresholds. Churners are the majority (58%), so maximising F1 would flag almost everyone (threshold 0.32 flags 78% of the players): the default is a balanced **0.5**, and the right value depends on what a retention action costs (lower = catch more churners, higher = fewer false alarms).

In [ ]:
oof = cross_val_predict(MODELS['random forest'][0], X, y, cv=StratifiedKFold(5, shuffle=True, random_state=SEED), method='predict_proba')[:, 1]
risk, churned = 1 - oof, 1 - y
prec, rec, thr = precision_recall_curve(churned, risk)
f1 = 2 * prec[:-1] * rec[:-1] / np.clip(prec[:-1] + rec[:-1], 1e-9, None)
THRESHOLD: float = 0.5
print(f'default churn-risk threshold: {THRESHOLD:.2f} (max-F1 threshold would be {float(thr[int(np.argmax(f1))]):.2f})')
table = []
for t in [0.3, 0.4, 0.5, 0.6, 0.7, 0.8]:
    pred = risk >= t
    table.append({'threshold': round(t, 2), 'players flagged': int(pred.sum()), 'precision': round((pred & (churned == 1)).sum() / max(pred.sum(), 1), 3), 'recall': round((pred & (churned == 1)).sum() / churned.sum(), 3)})
pd.DataFrame(table).drop_duplicates('threshold').sort_values('threshold')

## 3. Fit on all players and save

The saved file is a dict: the fitted pipeline, the feature list and labels, the median / range of every feature on the training players (defaults and sanity bounds for the app), the threshold and the evaluation. Reloading it and checking that the predictions are identical is the last cell.

In [ ]:
model = make_pipeline(SimpleImputer(strategy='median'), make_rf()).fit(X, y)
artifact: dict = {
    'model': model,
    'features': FEATURES,
    'feature_info': FEATURE_INFO,
    'defaults': X.median().to_dict(),
    'ranges': {c: (float(X[c].min()), float(X[c].max())) for c in FEATURES},
    'threshold_churn_risk': THRESHOLD,
    'metrics': df_eval.set_index('model').loc['random forest'].to_dict(),
    'trained_on': {'players': len(y), 'active_share': float(y.mean()), 'date': date.today().isoformat()},
    'note': 'Predicts P(still_active). Label is noisy (91 players flagged active are silent for 90+ days). Expect ROC-AUC about 0.8.',
}
joblib.dump(artifact, MODEL_PATH)

reloaded = joblib.load(MODEL_PATH)
same = np.allclose(reloaded['model'].predict_proba(X)[:, 1], model.predict_proba(X)[:, 1])
print(f'saved {MODEL_PATH} ({MODEL_PATH.stat().st_size / 1e6:.1f} MB) | reload gives identical predictions: {same}')
print('threshold:', round(reloaded['threshold_churn_risk'], 2), '| metrics:', reloaded['metrics'])

## 4. SHAP: what drives the predictions?

SHAP values of the saved random forest, expressed on the **churn risk** (red dot = high feature value; a point to the right raises the churn risk). They are computed on the training players, so they describe how the model behaves, not how well it generalises (see section 1 for that).

In [ ]:
import matplotlib.pyplot as plt
import shap

LABELS: list[str] = [FEATURE_INFO[f][0] for f in FEATURES]
imputer, forest = model.steps[0][1], model.steps[-1][1]
X_imp = pd.DataFrame(imputer.transform(X), columns=FEATURES)
sv = shap.TreeExplainer(forest)(X_imp)
churn_idx = list(forest.classes_).index(0)  # still_active = False
exp = shap.Explanation(values=sv.values[:, :, churn_idx], base_values=sv.base_values[:, churn_idx], data=X_imp.values, feature_names=LABELS)

shap.plots.beeswarm(exp, max_display=12, show=False)
plt.title('Impact on churn risk (training players)', loc='left')
plt.tight_layout()
plt.show()
shap.plots.bar(exp, max_display=12, show=False)
plt.title('Mean absolute impact on churn risk', loc='left')
plt.tight_layout()
plt.show()